# Long-horizon MuonClip-RMS baseline

Train or resume canonical Muon EMA + NS5 + exact RMS=0.20 matrix directions; QK clipping is N/A.

This notebook delegates training to the tested command-line API;
it does not define a private optimizer or model. Long runs are
opt-in so opening the notebook cannot accidentally start a
multi-day campaign. During training, the runtime also writes the
final 100 trained epoch-boundary model checkpoints to the dedicated
temporary cache selected by `CHECKPOINT_CACHE_ROOT`.


In [ ]:
# Papermill parameters. Override these values in an injected cell.
RUN_ROOT = ""
OUTPUT_ROOT = ""
CHECKPOINT_CACHE_ROOT = ""
CONFIG_PATH = ""
PROFILE = "pilot_1000_epochs"
PROTOCOL_SLUG = ""
SEEDS = [1337, 2027, 31415]
CHECKPOINT_PAYLOAD_CACHE_SIZE = 24
SHOW_PLOTS = True
REQUIRE_ARTIFACTS = True
ALLOW_TEMPORARY_LONG_RUN = False
CONFIG_PATH = ""
PROFILE = "pilot_1000_epochs"
OPTIMIZER_SLUG = "muonclip_rms"
EXECUTE_TRAINING = False
RESUME = True
REQUIRE_ARTIFACTS = False


In [ ]:
from pathlib import Path
from dataclasses import asdict, is_dataclass
from functools import lru_cache
import inspect
import json
import os
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (cwd, *cwd.parents)
        if (candidate / "baseline" / "rg_baselines").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find baseline/rg_baselines. Launch Jupyter from a clone of "
        "CalculatedContent/rg_optimizers."
    )
BASELINE_ROOT = REPO_ROOT / "baseline"
EXPERIMENT_ROOT = BASELINE_ROOT / "experiments" / "mnist_mlp3_tangent_rg"
if str(BASELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(BASELINE_ROOT))

default_root = os.environ.get(
    "RG_MNIST_TANGENT_ROOT", "/tmp/rg-mnist-mlp3-tangent-rg"
)
RUN_ROOT_PATH = Path(RUN_ROOT or default_root).expanduser().resolve()

default_checkpoint_cache_root = os.environ.get(
    "RG_MNIST_TANGENT_CHECKPOINT_CACHE_ROOT",
    "/tmp/rg-mnist-mlp3-tangent-checkpoints",
)
CHECKPOINT_CACHE_ROOT_PATH = Path(
    CHECKPOINT_CACHE_ROOT or default_checkpoint_cache_root
).expanduser().resolve()

def _suite_name_from_profile():
    if str(PROTOCOL_SLUG).strip():
        return str(PROTOCOL_SLUG).strip()
    candidate = (
        Path(CONFIG_PATH).expanduser()
        if str(CONFIG_PATH).strip()
        else EXPERIMENT_ROOT / "configs" / f"{PROFILE}.yaml"
    )
    if candidate.is_file():
        if candidate.suffix.lower() == ".json":
            payload = json.loads(candidate.read_text(encoding="utf-8"))
            value = payload.get("protocol", {}).get("suite_name")
            if value:
                return str(value)
        else:
            for line in candidate.read_text(encoding="utf-8").splitlines():
                stripped = line.strip()
                if stripped.startswith("suite_name:"):
                    return stripped.split(":", 1)[1].strip().strip("'\"")
    fallback = {
        "smoke": "mnist_mlp3_tangent_rg_v1_smoke",
        "pilot_1000_epochs": "mnist_mlp3_tangent_rg_v1_pilot1000",
        "long_horizon_10000_epochs": "mnist_mlp3_tangent_rg_v1_reference10000",
    }
    if PROFILE not in fallback:
        raise FileNotFoundError(
            f"Cannot derive suite_name for PROFILE={PROFILE!r}; set CONFIG_PATH "
            "or PROTOCOL_SLUG explicitly."
        )
    return fallback[PROFILE]

PROTOCOL_SLUG = _suite_name_from_profile()
OUTPUT_ROOT_PATH = Path(
    OUTPUT_ROOT or RUN_ROOT_PATH / PROTOCOL_SLUG / "notebook_outputs"
).expanduser().resolve()
OUTPUT_ROOT_PATH.mkdir(parents=True, exist_ok=True)

SEEDS = tuple(int(seed) for seed in SEEDS)
if SEEDS != (1337, 2027, 31415):
    print("WARNING: this is not the preregistered three-seed tuple:", SEEDS)

print("repository:", REPO_ROOT)
print("run root:", RUN_ROOT_PATH)
print("tail checkpoint cache root:", CHECKPOINT_CACHE_ROOT_PATH)
print("effective suite:", PROTOCOL_SLUG)
print("output root:", OUTPUT_ROOT_PATH)
print("seeds:", SEEDS)

EXPERIMENT_ROOT = BASELINE_ROOT / 'experiments' / 'mnist_mlp3_tangent_rg'


In [ ]:
from rg_baselines.statistics import summarize_numeric_metrics
from rg_baselines.tangent_rg import powerlaw_fit, trace_log


import subprocess

from rg_baselines.tangent_rg import (
    AdamWProfile,
    MuonClipRMSProfile,
    MuonProfile,
    TangentRGConfig,
    build_analysis_plan,
    list_analysis_checkpoints,
    list_capture_files,
    load_config,
    run_training,
)
from rg_baselines.tangent_rg.checkpoints import load_verified_tail_checkpoint_refs
from rg_baselines.tangent_rg.protocol import tail_checkpoint_epochs
from rg_baselines.tangent_rg import cli as tangent_cli


## Measured object

**`operator_kind`: `raw_weight_gram_esd`**

**`map_definition`: `For each layer W at a scheduled state, eigenvalues of the supported Gram matrix W W^T or W^T W.`**

**Identifiability caveat.** The raw weight ESD is the baseline observable. It is not a tangent-space Jacobian. Optimizer accuracy and late alpha/trace-log behavior validate the training baseline, not any quotient hypothesis.

These strings are persisted with every result row. A visually useful
spectrum does not change the identity of the map that produced it.


In [ ]:
def require_path(path, *, description="artifact"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}\n"
            "Run the prerequisite numbered notebook or set RUN_ROOT / "
            "OUTPUT_ROOT to the completed protocol directory."
        )
    return path


def first_existing(directory, names, *, description):
    directory = Path(directory)
    candidates = [directory / name for name in names]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {description} beneath {directory}. Expected one of:\n"
        + "\n".join(f"  - {path}" for path in candidates)
    )


def resolve_protocol_root():
    direct = RUN_ROOT_PATH / PROTOCOL_SLUG
    return direct if direct.is_dir() else RUN_ROOT_PATH


def resolve_arm_dir(optimizer_slug):
    protocol = resolve_protocol_root()
    candidates = [
        protocol / optimizer_slug,
        protocol / "results" / optimizer_slug,
        RUN_ROOT_PATH / optimizer_slug,
        RUN_ROOT_PATH / "results" / optimizer_slug,
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    if REQUIRE_ARTIFACTS:
        raise FileNotFoundError(
            f"No completed {optimizer_slug!r} arm was found. Checked:\n"
            + "\n".join(f"  - {path}" for path in candidates)
        )
    return candidates[0]


def resolve_seed_dir(optimizer_slug, seed):
    arm = resolve_arm_dir(optimizer_slug)
    candidates = [
        arm / f"seed_{int(seed)}",
        arm / f"seed_{int(seed):05d}",
        arm / "seeds" / f"seed_{int(seed)}",
        arm / "seeds" / f"seed_{int(seed):05d}",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Missing seed directory for optimizer={optimizer_slug}, seed={seed}. "
        f"Checked {candidates}."
    )


def validate_run_identity(seed_dir, *, optimizer_slug, seed):
    seed_dir = Path(seed_dir)
    manifest = json.loads(
        require_path(seed_dir / "manifest.json", description="run manifest")
        .read_text(encoding="utf-8")
    )
    resolved = json.loads(
        require_path(seed_dir / "resolved_config.json", description="resolved config")
        .read_text(encoding="utf-8")
    )
    completion = json.loads(
        require_path(seed_dir / "run_complete.json", description="completion marker")
        .read_text(encoding="utf-8")
    )
    config = dict(resolved.get("config", resolved))
    checks = {
        "manifest suite": (manifest.get("suite_name"), PROTOCOL_SLUG),
        "resolved suite": (config.get("suite_name"), PROTOCOL_SLUG),
        "manifest optimizer": (manifest.get("optimizer"), optimizer_slug),
        "resolved optimizer": (config.get("optimizer"), optimizer_slug),
        "completion optimizer": (completion.get("optimizer"), optimizer_slug),
        "manifest seed": (manifest.get("seed"), int(seed)),
        "resolved seed": (config.get("seed"), int(seed)),
        "completion seed": (completion.get("seed"), int(seed)),
    }
    mismatches = [
        f"{label}: observed={observed!r}, expected={expected!r}"
        for label, (observed, expected) in checks.items()
        if str(observed) != str(expected)
    ]
    fingerprints = {
        str(manifest.get("protocol_fingerprint", "")),
        str(resolved.get("protocol_fingerprint", "")),
        str(completion.get("protocol_fingerprint", "")),
    }
    if "" in fingerprints or len(fingerprints) != 1:
        mismatches.append(
            "manifest/resolved/completion protocol fingerprints are missing or unequal"
        )
    if not bool(completion.get("completed", False)):
        mismatches.append("run_complete.json does not declare completed=true")
    try:
        resolved_epochs = int(config["epochs"])
        completion_epochs = int(completion["epochs"])
        completion_step = int(completion["global_step"])
        best_validation_epoch = int(completion["best_validation_epoch"])
        analysis_plan = dict(resolved["analysis_plan"])
        plan_steps_per_epoch = int(analysis_plan["steps_per_epoch"])
        plan_total_steps = int(analysis_plan["total_steps"])
    except (KeyError, TypeError, ValueError) as error:
        mismatches.append(
            "resolved/completion final-horizon metadata is missing or invalid: "
            f"{type(error).__name__}: {error}"
        )
    else:
        if resolved_epochs < 1 or plan_steps_per_epoch < 1:
            mismatches.append("resolved epochs and steps_per_epoch must be positive")
        if plan_total_steps != resolved_epochs * plan_steps_per_epoch:
            mismatches.append(
                "resolved analysis_plan total_steps does not equal "
                "epochs * steps_per_epoch"
            )
        if completion_epochs != resolved_epochs:
            mismatches.append(
                f"completion epochs={completion_epochs} != resolved epochs={resolved_epochs}"
            )
        if completion_step != plan_total_steps:
            mismatches.append(
                f"completion global_step={completion_step} != resolved "
                f"analysis_plan total_steps={plan_total_steps}"
            )
        if not 0 <= best_validation_epoch <= resolved_epochs:
            mismatches.append(
                f"best_validation_epoch={best_validation_epoch} is outside "
                f"[0, {resolved_epochs}]"
            )
    if mismatches:
        raise RuntimeError(
            f"Run identity/provenance mismatch beneath {seed_dir}:\n  - "
            + "\n  - ".join(mismatches)
        )
    return manifest, resolved, completion


def validate_cross_run_provenance(manifests):
    manifests = list(manifests)
    if not manifests:
        raise RuntimeError("No manifests supplied for cross-run provenance audit")
    invariant_fields = (
        "suite_name", "dataset", "model", "initialization", "normalization",
        "train_indices_sha256", "validation_indices_sha256",
        "test_monitoring_only", "analysis_plan", "device", "software_versions",
        "determinism_settings",
    )
    disagreements = []
    for field in invariant_fields:
        serialized = {
            json.dumps(item.get(field), sort_keys=True, default=str)
            for item in manifests
        }
        if len(serialized) != 1:
            disagreements.append(field)
    if disagreements:
        raise RuntimeError(
            "Matched arms disagree on frozen run provenance fields: "
            + ", ".join(disagreements)
        )
    identities = {
        (str(item.get("optimizer")), int(item.get("seed"))) for item in manifests
    }
    expected = {
        (str(optimizer), int(seed))
        for optimizer in OPTIMIZER_SLUGS
        for seed in SEEDS
    } if "OPTIMIZER_SLUGS" in globals() else identities
    if identities != expected:
        raise RuntimeError(
            f"Manifest optimizer/seed grid is incomplete: observed={sorted(identities)}, "
            f"expected={sorted(expected)}"
        )
    return pd.DataFrame([
        {
            "optimizer": item.get("optimizer"),
            "seed": item.get("seed"),
            "device": item.get("device"),
            "software_versions": json.dumps(
                item.get("software_versions"), sort_keys=True, default=str
            ),
            "determinism_settings": json.dumps(
                item.get("determinism_settings"), sort_keys=True, default=str
            ),
            "pooling_compatibility_policy": (
                "headline pooling requires identical device, software versions, "
                "determinism settings, and scientific invariants across all runs"
            ),
        }
        for item in manifests
    ])


def record_dict(value):
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, dict):
        return dict(value)
    if hasattr(value, "__dict__"):
        return dict(vars(value))
    raise TypeError(f"Cannot convert {type(value).__name__} to an audit row")


def records_from_result(result):
    if result is None:
        return []
    if is_dataclass(result):
        return [record_dict(result)]
    if isinstance(result, dict):
        if "operator_kind" in result:
            return [dict(result)]
        rows = []
        for value in result.values():
            rows.extend(records_from_result(value))
        return rows
    if isinstance(result, (tuple, list)):
        rows = []
        for value in result:
            rows.extend(records_from_result(value))
        return rows
    return [record_dict(result)]


def spectrum_from_record(row):
    for name in (
        "spectrum", "eigenvalues", "singular_values", "rates",
        "positive_spectrum", "gram_spectrum",
    ):
        if name in row:
            values = np.asarray(row[name], dtype=float).reshape(-1)
            return values[np.isfinite(values) & (values > 0.0)]
    raise KeyError(
        "Operator record contains no recognized positive spectrum field. "
        f"Available fields: {sorted(row)}"
    )


def positive_spectrum(values, *, minimum_count=2):
    sample = np.asarray(values, dtype=float).reshape(-1)
    sample = sample[np.isfinite(sample) & (sample > 0.0)]
    sample = np.sort(sample)
    if sample.size < int(minimum_count):
        raise ValueError(
            f"Need at least {minimum_count} finite positive spectral values; "
            f"found {sample.size}."
        )
    return sample


def fit_spectrum_with_trace(
    values,
    *,
    operator_kind,
    map_definition,
    spectrum_kind,
    metadata,
    top_k_values=(0, 1, 2, 3, 4, 5),
    minimum_tail=8,
):
    # Fit amplitudes once, transform that fit to energy, and audit trace-log.
    # The power-law package is never called independently on squared values.
    # Trace-log uses squared values at the amplitude fit's independent rank.

    if str(spectrum_kind) != "amplitude":
        raise ValueError(
            "fit_spectrum_with_trace accepts operator amplitudes only; "
            "energy rows are produced by the exact amplitude-to-energy transform."
        )

    sample = positive_spectrum(values)
    feasible_top_k = tuple(
        int(value) for value in top_k_values if int(value) <= sample.size - 2
    )
    if not feasible_top_k or feasible_top_k[0] != 0:
        feasible_top_k = (0,)
    amplitude_fits = powerlaw_fit.fit_clipping_sensitivity(
        sample,
        top_k_values=feasible_top_k,
        minimum_tail=int(minimum_tail),
        operator_kind=str(operator_kind),
        map_definition=str(map_definition),
        spectrum_kind="amplitude",
        metadata=dict(metadata),
    )
    energy_rows = [
        powerlaw_fit.amplitude_fit_to_energy(row)
        for row in amplitude_fits.to_dict(orient="records")
    ]
    fits = pd.concat(
        [amplitude_fits, pd.DataFrame(energy_rows)],
        ignore_index=True,
        sort=False,
    )
    primary = amplitude_fits.loc[amplitude_fits["clip_top_k"].eq(0)].iloc[0]
    energy = sample ** 2
    trace_row = {
        **dict(metadata),
        "operator_kind": str(operator_kind),
        "map_definition": str(map_definition),
        "spectrum_kind": "energy_derived_from_amplitude",
        "support_rank_source": "powerlaw.Fit package-selected xmin tail count",
        "support_selected_from_same_trace_log": False,
        "support_rank": int(primary.get("n_tail", 0)),
        "trace_log_total": np.nan,
        "trace_log_per_eval": np.nan,
        "lambda_cut_scaled": np.nan,
        "trace_status": "fit_has_no_supported_tail",
    }
    rank = int(primary.get("n_tail", 0))
    if rank > 0:
        evaluated = trace_log.trace_log_at_rank(
            energy,
            rank=min(rank, energy.size),
            normalization_dimension=float(energy.size),
            rank_source="powerlaw.Fit package-selected xmin tail count",
        )
        trace_row.update(evaluated)
        trace_row["trace_status"] = "ok"
    return fits, pd.DataFrame([trace_row])


def save_analysis_frames(method_slug, *, operators, fits, traces):
    destination = OUTPUT_ROOT_PATH / "analyses" / str(method_slug)
    destination.mkdir(parents=True, exist_ok=True)
    required_identity = {
        "optimizer", "seed", "protocol_fingerprint", "source_artifact_kind"
    }
    for label, frame in (("operators", operators), ("fits", fits), ("traces", traces)):
        missing = required_identity - set(frame.columns)
        if missing:
            raise RuntimeError(
                f"{method_slug} {label} lack analysis provenance: {sorted(missing)}"
            )
        if frame[list(required_identity)].isna().any().any():
            raise RuntimeError(f"{method_slug} {label} contain null analysis provenance")
    identity_rows = fits[
        ["optimizer", "seed", "protocol_fingerprint", "source_artifact_kind"]
    ].drop_duplicates()
    duplicate_fingerprints = (
        identity_rows.groupby(["optimizer", "seed"], dropna=False)[
            "protocol_fingerprint"
        ].nunique()
    )
    if (duplicate_fingerprints != 1).any():
        raise RuntimeError(
            f"{method_slug} has multiple protocol fingerprints for one optimizer/seed"
        )
    fingerprint_grid = {
        f"{row.optimizer}:{int(row.seed)}": str(row.protocol_fingerprint)
        for row in identity_rows.itertuples(index=False)
    }
    expected_grid_count = identity_rows[["optimizer", "seed"]].drop_duplicates().shape[0]
    if len(fingerprint_grid) != expected_grid_count:
        raise RuntimeError(f"{method_slug} fingerprint-grid keys are not unique")
    provenance_manifest = {
        "schema_version": 1,
        "suite_name": str(PROTOCOL_SLUG),
        "method_slug": str(method_slug),
        "optimizer_seed_protocol_fingerprints": dict(sorted(fingerprint_grid.items())),
        "source_artifact_kinds": sorted(
            identity_rows["source_artifact_kind"].astype(str).unique().tolist()
        ),
        "operator_row_count": int(len(operators)),
        "fit_row_count": int(len(fits)),
        "trace_row_count": int(len(traces)),
    }
    operators.to_csv(destination / "operator_rows.csv", index=False)
    fits.to_csv(destination / "powerlaw_fits.csv", index=False)
    traces.to_csv(destination / "trace_log_independent_support.csv", index=False)
    (destination / "method_provenance.json").write_text(
        json.dumps(provenance_manifest, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    return destination


def save_spectrum_ccdf_gallery(
    spectral_arrays,
    *,
    method_slug,
    maximum_panels=24,
):
    # Save bounded log-log PDF/CCDF diagnostics for positive amplitudes.
    gallery = OUTPUT_ROOT_PATH / "analyses" / str(method_slug) / "spectrum_pdf_ccdf"
    gallery.mkdir(parents=True, exist_ok=True)
    rows = []
    for index, (key, raw) in enumerate(sorted(spectral_arrays.items())):
        if index >= int(maximum_panels):
            break
        sample = positive_spectrum(raw)
        x, ccdf = powerlaw_fit.empirical_ccdf(sample)
        fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.0))
        if sample[0] < sample[-1]:
            bins = np.geomspace(sample[0], sample[-1], min(50, max(8, sample.size // 3)))
            axes[0].hist(sample, bins=bins, density=True, histtype="step", linewidth=1.8)
        else:
            axes[0].scatter(sample, np.ones_like(sample), s=15)
        axes[1].step(x, ccdf, where="post", linewidth=1.8)
        for axis in axes:
            axis.set_xscale("log")
            axis.set_yscale("log")
            axis.grid(alpha=0.2)
        axes[0].set(xlabel="amplitude b", ylabel="density", title="PDF")
        axes[1].set(xlabel="amplitude b", ylabel="P(B >= b)", title="CCDF")
        fig.suptitle(str(key), fontsize=8)
        fig.tight_layout()
        safe = "".join(character if character.isalnum() or character in "-_" else "_" for character in str(key))
        path = gallery / f"{index:03d}_{safe[:160]}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        if SHOW_PLOTS and index < 3:
            plt.show()
        else:
            plt.close(fig)
        rows.append({"spectrum_key": str(key), "n_positive": int(sample.size), "figure": str(path)})
    index_frame = pd.DataFrame(rows)
    index_frame.to_csv(gallery / "index.csv", index=False)
    return index_frame


def plot_fit_alpha_ci(fits, *, method_slug, title):
    usable = fits.copy()
    if "fit_ok" in usable:
        usable = usable[boolean_series(usable["fit_ok"])]
    if "spectrum_kind" in usable:
        energy = usable[
            usable["spectrum_kind"].astype(str).eq("energy_derived_from_amplitude")
        ]
        if not energy.empty:
            usable = energy
    primary = usable[usable["clip_top_k"].eq(0)] if "clip_top_k" in usable else usable
    if primary.empty:
        raise RuntimeError(
            f"{method_slug}: no successful preregistered raw fits; inspect powerlaw_fits.csv"
        )
    if "state_index" not in primary:
        primary["state_index"] = 0
    groups = tuple(
        name
        for name in (
            "optimizer", "layer", "method", "null_kind", "pair_stride",
            "epsilon", "evidence_role",
        )
        if name in primary
    )
    if not groups:
        primary["method"] = str(method_slug)
        groups = ("method",)
    return plot_seed_ci(
        primary,
        x="state_index",
        metric="alpha",
        groups=groups,
        title=title,
        ylabel="Power-law density exponent alpha",
        reference=2.0,
        allow_incomplete=True,
        incomplete_output_path=(
            OUTPUT_ROOT_PATH / "analyses" / str(method_slug)
            / "incomplete_alpha_ci_groups.csv"
        ),
        output_path=(
            OUTPUT_ROOT_PATH / "analyses" / str(method_slug) / "alpha_95ci.png"
        ),
    )


def call_supported(function, /, *args, **kwargs):
    signature = inspect.signature(function)
    if any(
        parameter.kind is inspect.Parameter.VAR_KEYWORD
        for parameter in signature.parameters.values()
    ):
        return function(*args, **kwargs)
    supported = {key: value for key, value in kwargs.items() if key in signature.parameters}
    return function(*args, **supported)


def boolean_series(values):
    if getattr(values, "dtype", None) == bool:
        return values
    return values.astype(str).str.strip().str.lower().isin({"1", "true", "yes"})


def ci_summary(
    frame,
    *,
    groups,
    metrics,
    allow_incomplete=False,
    incomplete_output_path=None,
    return_incomplete=False,
):
    missing = set((*groups, *metrics, "seed")) - set(frame.columns)
    if missing:
        raise ValueError(f"CI input is missing columns: {sorted(missing)}")
    # Repeated layers/checkpoints/probes are not independent replicates.  First
    # collapse every declared group to one value per complete training seed.
    replicate = (
        frame.groupby([*groups, "seed"], as_index=False, dropna=False)[list(metrics)]
        .mean(numeric_only=True)
    )
    summary = summarize_numeric_metrics(
        replicate,
        group_columns=tuple(groups),
        metrics=tuple(metrics),
        confidence=0.95,
    )
    if summary.empty:
        raise RuntimeError("Confidence-interval summary is empty after seed aggregation")
    incomplete = summary[pd.to_numeric(summary["n"], errors="coerce") != len(SEEDS)]
    if not incomplete.empty:
        if incomplete_output_path is not None:
            incomplete_output_path = Path(incomplete_output_path)
            incomplete_output_path.parent.mkdir(parents=True, exist_ok=True)
            incomplete.to_csv(incomplete_output_path, index=False)
        if allow_incomplete:
            print(
                "WARNING: dropping incomplete CI identities from the mean/band; "
                "faint individual-seed traces remain visible.\n"
                + incomplete[
                    [name for name in (*groups, "metric", "n") if name in incomplete]
                ].to_string(index=False)
            )
        else:
            identity = [name for name in (*groups, "metric", "n") if name in incomplete]
            raise RuntimeError(
                "Every confidence-interval row requires exactly the preregistered "
                f"{len(SEEDS)} complete seeds. Incomplete identities:\n"
                + incomplete[identity].to_string(index=False)
            )
    complete = summary[pd.to_numeric(summary["n"], errors="coerce") == len(SEEDS)].copy()
    if return_incomplete:
        return complete, incomplete.copy()
    return complete


def plot_seed_ci(
    frame,
    *,
    x,
    metric,
    groups,
    title,
    ylabel,
    reference=None,
    output_path=None,
    allow_incomplete=False,
    incomplete_output_path=None,
):
    groups = tuple(groups)
    if allow_incomplete and incomplete_output_path is None and output_path is not None:
        output_path_for_report = Path(output_path)
        incomplete_output_path = output_path_for_report.with_name(
            output_path_for_report.stem + "_incomplete_ci_groups.csv"
        )
    summary, incomplete = ci_summary(
        frame,
        groups=(*groups, x),
        metrics=(metric,),
        allow_incomplete=allow_incomplete,
        incomplete_output_path=incomplete_output_path,
        return_incomplete=True,
    )
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    if not groups:
        frame = frame.copy()
        frame["series"] = "all"
        groups = ("series",)
    for identity, group in frame.groupby(list(groups), dropna=False):
        identity = identity if isinstance(identity, tuple) else (identity,)
        label = ", ".join(f"{key}={value}" for key, value in zip(groups, identity))
        for _, seed_frame in group.groupby("seed"):
            ordered = (
                seed_frame.groupby(x, as_index=False, dropna=False)[metric]
                .mean(numeric_only=True)
                .sort_values(x)
            )
            ax.plot(ordered[x], ordered[metric], alpha=0.16, linewidth=0.9)
        selected = summary.copy()
        for key, value in zip(groups, identity):
            selected = selected[selected[key].astype(str) == str(value)]
        selected = selected[selected["metric"] == metric].sort_values(x)
        if selected.empty:
            pass
        else:
            xv = selected[x].to_numpy(dtype=float)
            mean = selected["mean"].to_numpy(dtype=float)
            low = selected["ci_low"].to_numpy(dtype=float)
            high = selected["ci_high"].to_numpy(dtype=float)
            ax.plot(xv, mean, marker="o", linewidth=2.1, label=label)
            finite = np.isfinite(low) & np.isfinite(high)
            ax.fill_between(xv[finite], low[finite], high[finite], alpha=0.18)
        missing = incomplete.copy()
        for key, value in zip(groups, identity):
            missing = missing[missing[key].astype(str) == str(value)]
        missing = missing[missing["metric"] == metric]
        if not missing.empty:
            ax.scatter(
                missing[x].to_numpy(dtype=float),
                missing["mean"].to_numpy(dtype=float),
                marker="x", color="#555555", alpha=0.65, zorder=4,
            )
    if reference is not None:
        ax.axhline(float(reference), color="#333333", linestyle="--", linewidth=1.4)
    ax.set(xlabel=x, ylabel=ylabel, title=title)
    ax.set_xscale("symlog", linthresh=1.0)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    return summary, fig


def resolve_config_path(profile, explicit=""):
    if explicit:
        return require_path(explicit, description="training configuration")
    names = (f"{profile}.yaml", f"{profile}.yml", f"{profile}.json")
    roots = (
        EXPERIMENT_ROOT / "configs",
        BASELINE_ROOT / "rg_baselines" / "tangent_rg" / "configs",
        BASELINE_ROOT / "configs" / "mnist_mlp3_tangent_rg",
    )
    for root in roots:
        for name in names:
            candidate = root / name
            if candidate.is_file():
                return candidate
    epoch_by_profile = {
        "smoke": 2,
        "pilot_1000_epochs": 1_000,
        "long_horizon_10000_epochs": 10_000,
    }
    if profile not in epoch_by_profile:
        raise FileNotFoundError(
            f"No checked-in configuration for profile={profile!r}. Set CONFIG_PATH."
        )
    epochs = epoch_by_profile[profile]
    generated = OUTPUT_ROOT_PATH / "generated_configs" / f"{profile}.json"
    generated.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "protocol": {
            "suite_name": PROTOCOL_SLUG,
            "schema_version": 1,
            "description": f"Notebook-generated preregistered {profile} profile",
        },
        "training": {
            "epochs": epochs,
            "lr_schedule_epochs": min(30, epochs),
            "batch_size": 128,
            "validation_size": 5_000,
            "split_seed": 20_260_807,
            "validation_every_epochs": 1 if epochs == 2 else 5,
            "latest_every_epochs": 1,
            "test_monitoring_only": True,
        },
        "analysis": {
            "log_points": 2 if epochs == 2 else 96,
            "explicit_epochs": [0, 1, 2, 5, 10, 30],
            "dense_burst_anchor_epochs": [0, 1, 10, 100, 1_000],
            "dense_burst_length_steps": 4 if epochs == 2 else 8,
            "capture_parameter_names": ["fc1.weight", "fc2.weight"],
        },
        "runtime": {
            "device": "auto",
            "data_dir": str(RUN_ROOT_PATH / "data"),
            "run_root": str(RUN_ROOT_PATH),
            "tail_checkpoint_cache_root": str(CHECKPOINT_CACHE_ROOT_PATH),
        },
    }
    generated.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    checked = load_config(generated)
    checked.validate()
    print("generated validated config:", generated)
    return generated


def run_cli_training(*, optimizer, profile, seeds, resume):
    config_path = resolve_config_path(profile, CONFIG_PATH)
    launch_config = load_config(config_path)
    if str(launch_config.suite_name) != str(PROTOCOL_SLUG):
        raise RuntimeError(
            f"Resolved config suite_name={launch_config.suite_name!r} does not match "
            f"the notebook output suite {PROTOCOL_SLUG!r}. Clear PROTOCOL_SLUG or "
            "select the matching PROFILE/CONFIG_PATH."
        )
    temporary_root = Path("/tmp").resolve()
    if (
        int(launch_config.epochs) > 2
        and RUN_ROOT_PATH.is_relative_to(temporary_root)
        and not bool(ALLOW_TEMPORARY_LONG_RUN)
    ):
        raise RuntimeError(
            f"Refusing long-horizon training under temporary root {RUN_ROOT_PATH}. "
            "Set RUN_ROOT or RG_MNIST_TANGENT_ROOT to persistent storage. "
            "For a deliberate disposable run only, set "
            "ALLOW_TEMPORARY_LONG_RUN=True."
        )
    for seed in seeds:
        command = [
            sys.executable,
            "-m",
            "rg_baselines.tangent_rg.cli",
            "train",
            "--config",
            str(config_path),
            "--optimizer",
            str(optimizer),
            "--output-root",
            str(RUN_ROOT_PATH),
            "--tail-checkpoint-root",
            str(CHECKPOINT_CACHE_ROOT_PATH),
            "--seed",
            str(int(seed)),
        ]
        if resume:
            command.append("--resume")
        print("running:", " ".join(command))
        subprocess.run(command, cwd=BASELINE_ROOT, check=True)


def load_arm_outputs(optimizer):
    arm = resolve_arm_dir(optimizer)
    manifests = []
    performance_frames = []
    spectral_frames = []
    for seed in SEEDS:
        seed_dir = resolve_seed_dir(optimizer, seed)
        manifest, resolved, completion = validate_run_identity(
            seed_dir, optimizer_slug=optimizer, seed=seed
        )
        manifests.append(manifest)
        performance_path = first_existing(
            seed_dir / "metrics",
            (
                "performance_by_analysis_epoch.csv",
                "performance_by_analysis_state.csv",
            ),
            description="analysis-state performance table",
        )
        performance = pd.read_csv(performance_path)
        for column, expected in (("optimizer", optimizer), ("seed", int(seed))):
            if column not in performance:
                raise KeyError(f"{performance_path} lacks required identity column {column}")
            observed = set(performance[column].dropna().astype(str))
            if observed != {str(expected)}:
                raise RuntimeError(
                    f"{performance_path} identity mismatch for {column}: "
                    f"observed={sorted(observed)}, expected={expected}"
                )
        if "protocol_fingerprint" not in performance:
            raise KeyError(f"{performance_path} lacks protocol_fingerprint")
        performance_fingerprints = set(
            performance["protocol_fingerprint"].dropna().astype(str)
        )
        if performance_fingerprints != {str(manifest["protocol_fingerprint"])}:
            raise RuntimeError(
                f"{performance_path} fingerprint mismatch: "
                f"observed={sorted(performance_fingerprints)}"
            )
        performance["seed"] = int(seed)
        performance["optimizer"] = optimizer
        performance_frames.append(performance)
        spectral_path = first_existing(
            seed_dir / "metrics",
            (
                "spectral_metrics_by_analysis_epoch.csv",
                "weightwatcher_fits.csv",
                "weightwatcher_by_analysis_epoch.csv",
            ),
            description="raw + clip_xmax spectral fit table",
        )
        spectral = pd.read_csv(spectral_path)
        for column, expected in (("optimizer", optimizer), ("seed", int(seed))):
            if column not in spectral:
                raise KeyError(f"{spectral_path} lacks required identity column {column}")
            observed = set(spectral[column].dropna().astype(str))
            if observed != {str(expected)}:
                raise RuntimeError(
                    f"{spectral_path} identity mismatch for {column}: "
                    f"observed={sorted(observed)}, expected={expected}"
                )
        if "protocol_fingerprint" not in spectral:
            raise KeyError(f"{spectral_path} lacks protocol_fingerprint")
        spectral_fingerprints = set(
            spectral["protocol_fingerprint"].dropna().astype(str)
        )
        if spectral_fingerprints != {str(manifest["protocol_fingerprint"])}:
            raise RuntimeError(
                f"{spectral_path} fingerprint mismatch: "
                f"observed={sorted(spectral_fingerprints)}"
            )
        spectral["seed"] = int(seed)
        spectral["optimizer"] = optimizer
        spectral_frames.append(spectral)
    return (
        arm,
        manifests,
        pd.concat(performance_frames, ignore_index=True, sort=False),
        pd.concat(spectral_frames, ignore_index=True, sort=False),
    )


def canonical_columns(performance, spectral):
    performance = performance.copy()
    spectral = spectral.copy()
    for frame in (performance, spectral):
        if "analysis_epoch" in frame and "state_index" not in frame:
            frame["state_index"] = pd.to_numeric(
                frame["analysis_epoch"], errors="coerce"
            )
        if "analysis_state" in frame and "state_index" not in frame:
            frame["state_index"] = pd.to_numeric(
                frame["analysis_state"], errors="coerce"
            )
        if "epoch" in frame and "state_index" not in frame:
            frame["state_index"] = pd.to_numeric(frame["epoch"], errors="coerce")
    if "test_accuracy" not in performance and "test_acc" in performance:
        performance["test_accuracy"] = performance["test_acc"]
    if "layer" not in spectral and "layer_name" in spectral:
        spectral["layer"] = spectral["layer_name"]
    if "fit_variant" not in spectral:
        spectral["fit_variant"] = "unspecified"
    return performance, spectral


_VERIFIED_TAIL_CACHE_REFS = {}
_VERIFIED_TAIL_CHECKPOINT_IDENTITIES = {}
_VERIFIED_RUN_IDENTITIES = {}


def require_complete_seed(optimizer_slug, seed):
    seed_dir = resolve_seed_dir(optimizer_slug, seed)
    manifest, _, _ = validate_run_identity(
        seed_dir, optimizer_slug=optimizer_slug, seed=seed
    )
    _VERIFIED_RUN_IDENTITIES[(str(optimizer_slug), int(seed))] = {
        "protocol_fingerprint": str(manifest["protocol_fingerprint"]),
        "source_seed_dir": str(Path(seed_dir).resolve()),
    }
    return seed_dir


def verified_run_fingerprint(optimizer_slug, seed):
    identity = _VERIFIED_RUN_IDENTITIES.get((str(optimizer_slug), int(seed)))
    if identity is None:
        raise RuntimeError(
            f"Run identity was not verified for optimizer={optimizer_slug}, seed={seed}"
        )
    return str(identity["protocol_fingerprint"])


def require_tail_checkpoint_cache(optimizer_slug, seed):
    # Establish expected identity from the separately completed run. Never
    # trust identity claimed only by the temporary cache itself.
    source_seed_dir = resolve_seed_dir(optimizer_slug, seed)
    manifest, resolved, completion = validate_run_identity(
        source_seed_dir, optimizer_slug=optimizer_slug, seed=seed
    )
    _VERIFIED_RUN_IDENTITIES[(str(optimizer_slug), int(seed))] = {
        "protocol_fingerprint": str(manifest["protocol_fingerprint"]),
        "source_seed_dir": str(Path(source_seed_dir).resolve()),
    }
    resolved_values = dict(resolved.get("config", resolved))
    if "epochs" not in resolved_values:
        raise KeyError(f"Resolved run config lacks epochs: {source_seed_dir}")
    expected_epochs = tail_checkpoint_epochs(int(resolved_values["epochs"]))
    recorded_cache_root = Path(
        resolved_values.get(
            "tail_checkpoint_cache_root",
            "/tmp/rg-mnist-mlp3-tangent-checkpoints",
        )
    ).expanduser().resolve()
    if recorded_cache_root != CHECKPOINT_CACHE_ROOT_PATH:
        raise RuntimeError(
            "Notebook CHECKPOINT_CACHE_ROOT disagrees with the completed run: "
            f"notebook={CHECKPOINT_CACHE_ROOT_PATH}, recorded={recorded_cache_root}. "
            "Set CHECKPOINT_CACHE_ROOT (or "
            "RG_MNIST_TANGENT_CHECKPOINT_CACHE_ROOT) to the recorded cache root."
        )
    temporary_root = Path("/tmp").resolve()
    if (
        recorded_cache_root == temporary_root
        or not recorded_cache_root.is_relative_to(temporary_root)
    ):
        raise RuntimeError(
            f"Tail checkpoint cache must be a safe child of /tmp: {recorded_cache_root}"
        )
    cache_seed_dir = (
        recorded_cache_root
        / PROTOCOL_SLUG
        / str(optimizer_slug)
        / f"seed_{int(seed)}"
    ).resolve()
    refs = tuple(
        load_verified_tail_checkpoint_refs(
            cache_seed_dir,
            expected_suite_name=PROTOCOL_SLUG,
            expected_optimizer_name=str(optimizer_slug),
            expected_seed=int(seed),
            expected_fingerprint=str(manifest["protocol_fingerprint"]),
            expected_epochs=expected_epochs,
            validate_payloads=True,
        )
    )
    expected_count = min(100, int(resolved_values["epochs"]))
    if len(refs) != expected_count:
        raise RuntimeError(
            f"Verified cache count {len(refs)} != expected {expected_count}: "
            f"{cache_seed_dir}"
        )
    completed_epochs = int(completion.get("epochs", -1))
    completed_step = int(completion.get("global_step", -1))
    if completed_epochs != int(resolved_values["epochs"]) or completed_step < 1:
        raise RuntimeError(
            f"Completed run horizon is inconsistent beneath {source_seed_dir}"
        )
    steps_per_epoch, remainder = divmod(completed_step, completed_epochs)
    if remainder or steps_per_epoch < 1:
        raise RuntimeError(
            f"Completed step count is not an exact epoch grid beneath {source_seed_dir}"
        )
    expected_pairs = tuple(
        (int(epoch), int(epoch) * int(steps_per_epoch))
        for epoch in expected_epochs
    )
    observed_pairs = tuple((int(ref.epoch), int(ref.global_step)) for ref in refs)
    completion_cache_dir = completion.get("tail_checkpoint_cache_dir")
    if not isinstance(completion_cache_dir, str) or not completion_cache_dir.strip():
        raise RuntimeError(
            f"Completion marker lacks tail_checkpoint_cache_dir: {source_seed_dir}"
        )
    source_cache_checks = {
        "cache_dir": (
            str(Path(completion_cache_dir).expanduser().resolve()),
            str(cache_seed_dir),
        ),
        "checkpoint_count": (completion.get("tail_checkpoint_count"), expected_count),
        "first_epoch": (completion.get("tail_checkpoint_first_epoch"), expected_epochs[0]),
        "last_epoch": (completion.get("tail_checkpoint_last_epoch"), expected_epochs[-1]),
        "epoch_step_grid": (observed_pairs, expected_pairs),
    }
    mismatches = [
        f"{label}: observed={observed!r}, expected={expected!r}"
        for label, (observed, expected) in source_cache_checks.items()
        if observed != expected
    ]
    if mismatches:
        raise RuntimeError(
            "Tail cache disagrees with the persistent run completion marker:\n  - "
            + "\n  - ".join(mismatches)
        )
    resolved_cache_dir = cache_seed_dir.resolve()
    _VERIFIED_TAIL_CACHE_REFS[resolved_cache_dir] = refs
    for ref in refs:
        _VERIFIED_TAIL_CHECKPOINT_IDENTITIES[ref.path.resolve()] = {
            "protocol_fingerprint": str(manifest["protocol_fingerprint"]),
            "optimizer": str(optimizer_slug),
            "seed": int(seed),
            "epoch": int(ref.epoch),
            "global_step": int(ref.global_step),
            "cache_seed_dir": str(resolved_cache_dir),
        }
    return resolved_cache_dir


## Launch or resume

Set `CONFIG_PATH` to a resolved `smoke`, `pilot_1000_epochs`, or
`long_horizon_10000_epochs` configuration and set
`EXECUTE_TRAINING=True`. The exact command is printed before each
seeded run. `CHECKPOINT_CACHE_ROOT` defaults to
`/tmp/rg-mnist-mlp3-tangent-checkpoints` and may also be set with
`RG_MNIST_TANGENT_CHECKPOINT_CACHE_ROOT`; it must remain a safe
child of `/tmp`. The cache is populated online, independently of
WeightWatcher cadence, and is never reconstructed by an analysis
notebook. Keep this cache through notebooks `10`, `12`, `13`, and
`15`, or make an external byte-for-byte backup: losing it after
completion requires full `--overwrite` retraining because
`--resume` cannot recreate past epoch boundaries.


In [ ]:
if EXECUTE_TRAINING:
    run_cli_training(
        optimizer=OPTIMIZER_SLUG,
        profile=PROFILE,
        seeds=SEEDS,
        resume=bool(RESUME),
    )
else:
    print(
        "Training not launched. Set EXECUTE_TRAINING=True after "
        "checking CONFIG_PATH, RUN_ROOT, and persistent storage."
    )


## Artifact completeness and protocol audit


In [ ]:
if EXECUTE_TRAINING or REQUIRE_ARTIFACTS:
    arm_dir, manifests, performance, spectral = load_arm_outputs(
        OPTIMIZER_SLUG
    )
    performance, spectral = canonical_columns(performance, spectral)
    observed_seeds = set(pd.to_numeric(performance["seed"]).astype(int))
    assert observed_seeds == set(SEEDS), (observed_seeds, SEEDS)
    if "test_monitoring_only" in performance:
        assert performance["test_monitoring_only"].astype(int).eq(1).all()
    tail_cache_rows = []
    for seed in SEEDS:
        seed_dir = resolve_seed_dir(OPTIMIZER_SLUG, seed)
        checkpoints_dir = seed_dir / "checkpoints"
        for name in (
            "checkpoint_latest.pt",
            "checkpoint_best.pt",
            "checkpoint_final.pt",
        ):
            require_path(checkpoints_dir / name, description=name)
        refs = list_analysis_checkpoints(checkpoints_dir)
        if not refs or refs[0].epoch != 0 or refs[0].global_step != 0:
            raise RuntimeError(
                f"{seed_dir} is missing its immutable epoch-zero analysis checkpoint"
            )
        cache_seed_dir = require_tail_checkpoint_cache(
            OPTIMIZER_SLUG, seed
        )
        cache_manifest = json.loads(
            (cache_seed_dir / "manifest.json").read_text(encoding="utf-8")
        )
        cache_completion = json.loads(
            (cache_seed_dir / "cache_complete.json").read_text(encoding="utf-8")
        )
        tail_cache_rows.append({
            "optimizer": OPTIMIZER_SLUG,
            "seed": int(seed),
            "cache_seed_dir": str(cache_seed_dir),
            "checkpoint_count": int(cache_completion["checkpoint_count"]),
            "first_epoch": int(cache_completion["first_epoch"]),
            "last_epoch": int(cache_completion["last_epoch"]),
            "protocol_fingerprint": cache_manifest["protocol_fingerprint"],
            "completed": bool(cache_completion["completed"]),
        })
    print("complete arm:", arm_dir)
    display(pd.DataFrame(manifests))
    display(pd.DataFrame(tail_cache_rows))
    display(performance.tail(12))
    display(spectral.tail(18))
else:
    performance = spectral = pd.DataFrame()
    print("Artifact audit deferred because REQUIRE_ARTIFACTS=False.")


## Performance and fixed-point plots with complete-run uncertainty


In [ ]:
if not performance.empty:
    perf_summary, _ = plot_seed_ci(
        performance,
        x="state_index",
        metric="test_accuracy",
        groups=("optimizer",) if "optimizer" in performance else (),
        title=f"{OPTIMIZER_SLUG}: monitoring-only test accuracy",
        ylabel="Test accuracy",
        output_path=OUTPUT_ROOT_PATH / OPTIMIZER_SLUG / "test_accuracy_95ci.png",
    )
    valid = spectral.copy()
    if "status" in valid:
        valid = valid[valid["status"].astype(str).eq("ok")]
    if "selection_role" in valid:
        preferred = valid[
            valid["selection_role"].astype(str).isin(
                ["primary", "preregistered_primary"]
            )
        ]
        if not preferred.empty:
            valid = preferred
    alpha_summary, _ = plot_seed_ci(
        valid,
        x="state_index",
        metric="alpha",
        groups=("layer", "fit_variant"),
        title=f"{OPTIMIZER_SLUG}: layerwise alpha",
        ylabel="Power-law density exponent alpha",
        reference=2.0,
        allow_incomplete=True,
        incomplete_output_path=(
            OUTPUT_ROOT_PATH / OPTIMIZER_SLUG
            / "alpha_incomplete_ci_groups.csv"
        ),
        output_path=OUTPUT_ROOT_PATH / OPTIMIZER_SLUG / "alpha_95ci.png",
    )
    perf_summary.to_csv(
        OUTPUT_ROOT_PATH / OPTIMIZER_SLUG / "performance_notebook_95ci.csv",
        index=False,
    )
    alpha_summary.to_csv(
        OUTPUT_ROOT_PATH / OPTIMIZER_SLUG / "alpha_notebook_95ci.csv",
        index=False,
    )


## Interpretation gate

High accuracy and a late alpha near two establish only the baseline
regime. Inspect KS distance, tail count, tail decades, standard
versus `clip_xmax`, and the independently supported trace-log rows
before describing a layer as fixed-point-like. FC3 has at most ten
positive modes and cannot carry the main power-law conclusion.
